# LOAD DATA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/My Drive/ML/lab3ImageSegmentation/test_images.zip" /content/
!cp "/content/drive/My Drive/ML/lab3ImageSegmentation/full_dataset.zip" /content/

In [ ]:
!unzip -q /content/test_images.zip -d /content/
!unzip -q /content/full_dataset.zip -d /content/

# IMPORTS

In [ ]:
!pip install segmentation_models_pytorch
!pip install -q albumentations

In [ ]:
import json
import os
import random
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from segmentation_models_pytorch.encoders import get_preprocessing_fn
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# CONFIG

In [ ]:
DATA_ROOT = Path(r"/content/full_dataset/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 384
BATCH_SIZE = 12
NUM_EPOCHS = 60
EARLY_STOPPING_CONST = 10
WARMUP_EPOCHS = 6
FROZEN_EPOCH = 3
LR = 3e-4
WEIGHT_DECAY = 5e-4
N_SPLITS = 5
TRAIN_FOLDS = [1, 2, 3, 4, 5]
NUM_WORKERS = 1
SEED = 42

MODEL_ARCH = "UnetPlusPlus"
ENCODER_NAME = "timm-efficientnet-b3"
ENCODER_WEIGHTS = "noisy-student"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# UTILS

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def compute_dice(probs, targets, threshold=0.5, eps=1e-7):
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)
    return ((2.0 * intersection + eps) / (denom + eps)).mean().item()

def set_frozen(model, encoder_frozen=True):
    for param in model.encoder.parameters():
        param.requires_grad = not encoder_frozen

    if encoder_frozen:
        print("--- Энкодер ЗАМОРОЖЕН (учится только декодер) ---")
    else:
        print("--- Энкодер РАЗМОРОЖЕН (полное обучение модели) ---")

def plot_training_results(history, fold, save_dir):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(16, 10))
    plt.suptitle(f"Training Results - Fold {fold}", fontsize=20)

    # 1. График Loss (Train vs Val)
    plt.subplot(2, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss', color='blue', lw=2)
    plt.plot(epochs, history['val_loss'], label='Val Loss', color='red', lw=2)
    plt.title('Loss Dynamics')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 2. График Dice Score
    plt.subplot(2, 2, 2)
    plt.plot(epochs, history['val_dice'], label='Val Dice', color='green', lw=2)
    plt.axhline(y=max(history['val_dice']), color='gray', linestyle='--', alpha=0.5)
    plt.title(f'Best Dice: {max(history["val_dice"]):.4f}')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 3. График Learning Rate
    plt.subplot(2, 2, 3)
    plt.plot(epochs, history['lr'], label='LR', color='orange', lw=2)
    plt.yscale('log') # Логарифмическая шкала удобнее для LR
    plt.title('Learning Rate (Log Scale)')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 4. График Threshold
    plt.subplot(2, 2, 4)
    plt.scatter(epochs, history['th'], label='Best Threshold', color='purple', alpha=0.6)
    plt.plot(epochs, history['th'], color='purple', alpha=0.3)
    plt.title('Best Threshold Per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(save_dir / f"fold_{fold}_history.png", dpi=150)
    plt.show()

# AUGMENTATION

In [ ]:
def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.ShiftScaleRotate(shift_limit=0.01, scale_limit=0.05, rotate_limit=5, p=0.1),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=25, val_shift_limit=25, p=0.5),
        A.Blur(blur_limit=3, p=0.1),
        A.ShotNoise(
            scale_range = (0.01, 0.1),
            p= 0.1
        ),
        A.CoarseDropout(
            num_holes_range=(1,8),
            hole_height_range=(0., 0.1),
            hole_width_range=(0., 0.1),
            fill_mask=0,
            p=0.1
        ),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size)
    ])

In [ ]:
def visualize_augmentations(images_dir, masks_dir, img_size=384, num_samples=3):
    print("\n--- Визуализация аугментаций ---")
    transforms = get_train_transforms(img_size)

    mask_paths = list(Path(masks_dir).glob("*.png"))
    if not mask_paths:
        print("Маски не найдены!")
        return

    sampled_masks = random.sample(mask_paths, min(num_samples, len(mask_paths)))

    fig, axes = plt.subplots(len(sampled_masks), 3, figsize=(15, 5 * len(sampled_masks)))
    if len(sampled_masks) == 1:
        axes = [axes]

    for i, mask_path in enumerate(sampled_masks):
        stem = mask_path.stem

        # Ищем соответствующую картинку
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".webp"]:
            p = Path(images_dir) / f"{stem}{ext}"
            if p.exists():
                img_path = p
                break

        if img_path is None:
            continue

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.uint8)

        augmented = transforms(image=image, mask=mask)
        aug_image = augmented['image']
        aug_mask = augmented['mask']

        # Оригинал
        axes[i][0].imshow(image)
        axes[i][0].set_title(f"Оригинал\n{stem[:15]}...")
        axes[i][0].axis('off')

        # Аугментированное фото
        axes[i][1].imshow(aug_image)
        axes[i][1].set_title("Аугментация (Цвет, Шум, Dropout)")
        axes[i][1].axis('off')

        # Аугментированная маска
        axes[i][2].imshow(aug_mask, cmap='gray')
        axes[i][2].set_title("Аугментированная маска")
        axes[i][2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_augmentations(IMAGES_DIR, MASKS_DIR, IMG_SIZE, num_samples=3)

# CLASS

In [ ]:
class SupermarketDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms, preprocess_fn=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transforms = transforms
        self.preprocess_fn = preprocess_fn

        self.samples = []
        self.ips = [] 

        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            # Ищем картинку
            image_path = None
            for ext in [".jpg", ".jpeg", ".png", ".webp"]:
                p = self.images_dir / f"{stem}{ext}"
                if p.exists():
                    image_path = p
                    break

            if image_path:
                self.samples.append((image_path, mask_path))
                # Извлекаем IP платформы из имени 
                ip = stem.split('_')[0]
                self.ips.append(ip)

        print(f"Найдено {len(self.samples)} пар. Уникальных платформ: {len(set(self.ips))}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        # albumentations
        augmented = self.transforms(image=image, mask=mask)
        image = augmented['image']
        mask = augmented['mask']

        # Preprocessing 
        if self.preprocess_fn:
            image = self.preprocess_fn(image)
        else:
            image = image / 255.0

        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None, ...]).float()

        return image, mask

class ComboLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode=smp.losses.BINARY_MODE, from_logits=True)

    def forward(self, logits, targets):
        return 0.4 * self.bce(logits, targets) + 0.6 * self.dice(logits, targets)

class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_dice):
        if self.best_score is None:
            self.best_score = val_dice
        elif val_dice < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"--- EarlyStopping counter: {self.counter} out of {self.patience} ---")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_dice
            self.counter = 0

# TRAIN / VALIDATION

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device, encoder_frozen=False):
    model.train()
    running_loss = 0.0

    if encoder_frozen:
        model.encoder.eval()

    pbar = tqdm(loader, desc="Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader)

@torch.no_grad()
def valid_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_masks = []

    for images, masks in tqdm(loader, desc="Valid", leave=False):
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        loss = loss_fn(logits, masks)

        running_loss += loss.item()

        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_masks.append(masks.cpu())

    all_probs = torch.cat(all_probs)
    all_masks = torch.cat(all_masks)

    # Ищем лучший порог для DICE
    best_dice = 0
    best_th = 0.5
    for th in np.linspace(0.3, 0.7, 17):
        dice = compute_dice(all_probs, all_masks, threshold=th)
        if dice > best_dice:
            best_dice = dice
            best_th = th

    return running_loss / len(loader), best_dice, best_th

# MAIN

In [ ]:
def main():
    seed_everything(SEED)

    preprocess_fn = get_preprocessing_fn(ENCODER_NAME, pretrained=ENCODER_WEIGHTS)

    base_dataset = SupermarketDataset(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR,
        transforms=get_val_transforms(IMG_SIZE), # Временные
        preprocess_fn=preprocess_fn
    )

    ips = np.array(base_dataset.ips)

    # Используем StratifiedKFold по IP весов
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_dice': [],
        'lr': [],
        'th': []
    }
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(ips)), ips), 1):
        if fold not in TRAIN_FOLDS:
            continue

        print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")

        # Разделяем данные
        train_samples = [base_dataset.samples[i] for i in train_idx]
        val_samples = [base_dataset.samples[i] for i in val_idx]

        # Инициализируем датасеты с правильными аугментациями
        train_dataset = SupermarketDataset(IMAGES_DIR, MASKS_DIR, get_train_transforms(IMG_SIZE), preprocess_fn)
        train_dataset.samples = train_samples

        val_dataset = SupermarketDataset(IMAGES_DIR, MASKS_DIR, get_val_transforms(IMG_SIZE), preprocess_fn)
        val_dataset.samples = val_samples

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        # Модель
        model = smp.create_model(
            arch=MODEL_ARCH,
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            decoder_attention_type = "scse",
            in_channels=3,
            classes=1,
            activation=None
        ).to(DEVICE)

        loss_fn = ComboLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        scheduler_warmup = LinearLR(optimizer, start_factor=0.005, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        scheduler_cosine = CosineAnnealingLR(optimizer, T_max=(NUM_EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)

        best_val_dice = -1.0
        early_stopping = EarlyStopping(patience=EARLY_STOPPING_CONST, verbose=True)

        # Замораживаем в самом начале
        set_frozen(model, encoder_frozen=True)
        is_currently_frozen = True

        for epoch in range(1, NUM_EPOCHS + 1):
            # Размораживаем
            if is_currently_frozen and epoch > FROZEN_EPOCH :
                set_frozen(model, encoder_frozen=False)
                is_currently_frozen = False

            train_loss = train_epoch(model, train_loader, optimizer, loss_fn, DEVICE, encoder_frozen=is_currently_frozen)
            val_loss, val_dice, best_th = valid_epoch(model, val_loader, loss_fn, DEVICE)

            current_lr = optimizer.param_groups[0]['lr']
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_dice'].append(val_dice)
            history['lr'].append(current_lr)
            history['th'].append(best_th)

            # Шагаем планировщиком
            if epoch <= WARMUP_EPOCHS:
                scheduler_warmup.step()
            else:
                scheduler_cosine.step()

            print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val DICE: {val_dice:.4f} (Th: {best_th:.5f}) | LR: {current_lr:.6f}")

            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "threshold": best_th,
                    "val_dice": val_dice,
                    "config": {
                        "MODEL_NAME": MODEL_ARCH,
                        "ENCODER_NAME": ENCODER_NAME,
                        "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                        "IMG_SIZE": IMG_SIZE
                    }
                }, SAVE_DIR / f"best_model_fold_{fold}.pth")
                print(f"-> Модель сохранена! DICE улучшился.")

            early_stopping(val_dice)
            if early_stopping.early_stop:
                print(f"Early stopping на эпохе {epoch}!")
                break

        plot_training_results(history, fold, SAVE_DIR)
        pd.DataFrame(history).to_csv(SAVE_DIR / f"history_fold_{fold}.csv", index=False)
        fold_results.append(best_val_dice)

    print(f"\nСредний DICE по {TRAIN_FOLDS} фолдам: {np.mean(fold_results):.4f}")

In [ ]:
main()

# CONFIG INF

In [ ]:
# !cp "/content/drive/MyDrive/ML/lab3ImageSegmentation/weights/best_fold_1.pth" /content/seg_train_runs
# !cp "/content/drive/MyDrive/ML/lab3ImageSegmentation/weights/best_fold_2.pth" /content/seg_train_runs
# !cp "/content/drive/MyDrive/ML/lab3ImageSegmentation/weights/best_fold_3.pth" /content/seg_train_runs
# !cp "/content/drive/MyDrive/ML/lab3ImageSegmentation/weights/best_fold_4.pth" /content/seg_train_runs
# !cp "/content/drive/MyDrive/ML/lab3ImageSegmentation/weights/best_fold_5.pth" /content/seg_train_runs

In [ ]:
TEST_IMAGES_DIR = Path(r"/content/test_images")
OUTPUT_CSV = "submission.csv"
CHECKPOINTS_DIR = Path(r"/content/seg_train_runs")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
CHECKPOINTS_TO_LOAD = [
    "best_model_fold_1.pth",
    "best_model_fold_2.pth",
    "best_model_fold_3.pth",
    "best_model_fold_4.pth",
    "best_model_fold_5.pth"
]

# UTILS INF


In [ ]:
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)

def build_model(arch: str, encoder_name: str, encoder_weights: str=None) -> nn.Module:
    model = smp.create_model(
        arch=arch,
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        decoder_attention_type = "scse",
        in_channels=3,
        classes=1,
        activation=None
    )
    return model

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))

def predict_with_flip_tta(model, img_tensor):
    with torch.no_grad():
        logits_orig = model(img_tensor)
        prob_orig = torch.sigmoid(logits_orig)

    img_flipped = torch.flip(img_tensor, dims=[3])

    with torch.no_grad():
        logits_flipped = model(img_flipped)
        prob_flipped = torch.sigmoid(logits_flipped)

    prob_flipped_back = torch.flip(prob_flipped, dims=[3])

    avg_prob = (prob_orig + prob_flipped_back) / 2.0

    return avg_prob

# INIT MODELS


In [ ]:
ensemble_models = []

print("Загрузка моделей ансамбля...")
for i, filename in enumerate(CHECKPOINTS_TO_LOAD, 1):
    ckpt_path = CHECKPOINTS_DIR / filename
    if not ckpt_path.exists():
        print(f"Файл {ckpt_path} не найден. Пропуск.")
        continue

    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = checkpoint.get("config", {})
    th = checkpoint.get("threshold", 0.5)

    # Берем настройки из конфига 
    model_name = cfg.get("MODEL_NAME", "Unet")
    encoder_name = cfg.get("ENCODER_NAME", "timm-efficientnet-b3")
    img_size = int(cfg.get("IMG_SIZE", 384))

    model = build_model(model_name, encoder_name, encoder_weights=None)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    enc_weights = cfg.get("ENCODER_WEIGHTS", "noisy-student")
    prep_fn = get_preprocessing_fn(encoder_name, pretrained=enc_weights) if enc_weights else None

    ensemble_models.append({
        "model": model,
        "prep_fn": prep_fn,
        "img_size": img_size,
        "threshold": th
    })
    print(f"Модель {i}. THRESHOLD = {th}")

if not ensemble_models:
    raise RuntimeError("Ни одна модель не загружена! Проверь пути к весам.")

print(f"Всего моделей в ансамбле: {len(ensemble_models)}")

# INFERENCE

In [ ]:
image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

if not image_paths:
    raise FileNotFoundError(f"No images found in: {TEST_IMAGES_DIR}")

print(f"Найдено тестовых изображений: {len(image_paths)}")

rows = []

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Инференс"):
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"\n[Пропуск] Не удалось прочитать: {img_path}")
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        ensemble_probs = np.zeros((H, W), dtype=np.float32)

        for i, m_dict in enumerate(ensemble_models, 1):
            model = m_dict["model"]
            prep_fn = m_dict["prep_fn"]
            size = m_dict["img_size"]
            model_th = m_dict["threshold"]

            inp = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
            inp = inp.astype(np.float32)

            if prep_fn is not None:
                inp = prep_fn(inp)
            else:
                inp = inp / 255.0

            # HWC -> CHW -> BCHW [1, 3, size, size]
            inp_tensor = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

            # TTA
            prob_tensor = predict_with_flip_tta(model, inp_tensor)

            # [1, 1, size, size] -> [size, size]
            pred = prob_tensor[0, 0].cpu().numpy()

            if pred.shape != (H, W):
                pred = cv2.resize(pred.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)

            ensemble_probs += (pred - model_th)

        ensemble_probs /= len(ensemble_models)
        mask = (ensemble_probs > 0.0).astype(np.uint8)

        rows.append(
            {
                "ImageId": img_path.name,
                "mask": serialize_mask(mask),
            }
        )

# SAVE

In [ ]:
submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("\nГотово!")
print(f"Результат сохранен в: {OUTPUT_CSV}")
print(submission_df.head(3))